## Setup

In [ ]:
# !pip install tokenizers

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, processors, decoders, trainers
from tokenizers.models import BPE, WordPiece, Unigram
from tokenizers.normalizers import NFD, NFKC, Lowercase, StripAccents, Replace, Strip, Sequence as NormSeq
from tokenizers.pre_tokenizers import Whitespace, WhitespaceSplit, ByteLevel, CharDelimiterSplit, Digits, Punctuation, Sequence as PreSeq
from tokenizers.processors import TemplateProcessing
from tokenizers.decoders import ByteLevel as ByteLevelDecoder, WordPiece as WordPieceDecoder

print("✅ Imports successful")

---

# Part 1: Normalization

Normalization cleans and standardizes text before tokenization.

## 1.1 Unicode Normalization (NFD, NFC, NFKC, NFKD)

**NFD** (Canonical Decomposition): Breaks characters into components
**NFC** (Canonical Composition): Combines components
**NFKC** (Compatibility Composition): Normalizes to standard forms

In [ ]:
# Test text with special characters
test_text = "Héllo Wörld! café résumé"

# NFD - Decompose
nfd = normalizers.NFD()
print(f"Original:  {test_text}")
print(f"NFD:       {nfd.normalize_str(test_text)}")
print(f"Bytes:     {[c for c in nfd.normalize_str(test_text)]}")

# NFKC - Compose and compatibility normalize
nfkc = normalizers.NFKC()
print(f"\nNFKC:      {nfkc.normalize_str(test_text)}")

## 1.2 Lowercase Normalization

In [ ]:
lowercase = normalizers.Lowercase()

examples = [
    "Hello World",
    "MACHINE LEARNING",
    "NaTuRaL LaNgUaGe"
]

print("Lowercase Normalization:")
print("=" * 60)
for text in examples:
    normalized = lowercase.normalize_str(text)
    print(f"{text:25s} → {normalized}")

## 1.3 Strip Accents

In [ ]:
strip_accents = normalizers.StripAccents()

examples = [
    "café résumé",
    "naïve señor",
    "Москва",  # Moscow in Russian
    "北京"      # Beijing in Chinese
]

print("Strip Accents:")
print("=" * 60)
for text in examples:
    normalized = strip_accents.normalize_str(text)
    print(f"{text:25s} → {normalized}")

## 1.4 Replace Normalizer (Pattern Replacement)

In [ ]:
# Replace multiple spaces with single space
replace_spaces = normalizers.Replace("  +", " ")

examples = [
    "Hello    world",
    "Too     many       spaces",
    "Normal text"
]

print("Replace Multiple Spaces:")
print("=" * 60)
for text in examples:
    normalized = replace_spaces.normalize_str(text)
    print(f"'{text:30s}' → '{normalized}'")

# Remove URLs
remove_urls = normalizers.Replace(r"https?://\S+", "[URL]")
text_with_url = "Check out https://huggingface.co for more info"
print(f"\nRemove URLs:")
print(f"Original:   {text_with_url}")
print(f"Normalized: {remove_urls.normalize_str(text_with_url)}")

## 1.5 Strip Normalizer (Remove Leading/Trailing Whitespace)

In [ ]:
strip = normalizers.Strip()

examples = [
    "  leading spaces",
    "trailing spaces  ",
    "  both sides  ",
    "\ttabs and newlines\n"
]

print("Strip Whitespace:")
print("=" * 60)
for text in examples:
    normalized = strip.normalize_str(text)
    print(f"'{text}' → '{normalized}'")

## 1.6 Combining Normalizers (Sequence)

Chain multiple normalizers together:

In [ ]:
# BERT-style normalization
bert_normalizer = normalizers.Sequence([
    normalizers.NFD(),
    normalizers.Lowercase(),
    normalizers.StripAccents(),
])

test_texts = [
    "HELLO WORLD!",
    "Café Résumé",
    "   Spaced   Out   "
]

print("BERT-style Normalization (NFD + Lowercase + StripAccents):")
print("=" * 60)
for text in test_texts:
    normalized = bert_normalizer.normalize_str(text)
    print(f"{text:30s} → {normalized}")

## 1.7 Real-World Example: Clean Social Media Text

In [ ]:
# Social media text normalizer
social_media_normalizer = normalizers.Sequence([
    normalizers.Replace(r"@\w+", "[USER]"),           # Replace mentions
    normalizers.Replace(r"#\w+", "[HASHTAG]"),        # Replace hashtags
    normalizers.Replace(r"https?://\S+", "[URL]"),    # Replace URLs
    normalizers.Replace(r"  +", " "),                 # Multiple spaces
    normalizers.Strip(),                               # Trim
    normalizers.Lowercase()                            # Lowercase
])

social_texts = [
    "Hey @john check out https://example.com #awesome",
    "RT @user: This    is   cool! #AI #ML",
    "  Follow me @alice for more #content  "
]

print("Social Media Text Normalization:")
print("=" * 60)
for text in social_texts:
    normalized = social_media_normalizer.normalize_str(text)
    print(f"Original:   {text}")
    print(f"Normalized: {normalized}\n")

---

# Part 2: Pre-tokenization

Pre-tokenization splits text into words or subword units **before** the model processes it.

## 2.1 Whitespace Pre-tokenizer

In [ ]:
whitespace = pre_tokenizers.Whitespace()

text = "Hello world! How are you?"
pre_tokenized = whitespace.pre_tokenize_str(text)

print(f"Text: {text}")
print(f"\nPre-tokenized (word, offset):")
for word, offset in pre_tokenized:
    print(f"  '{word}' at {offset}")

## 2.2 WhitespaceSplit vs Whitespace

**Whitespace**: Splits on whitespace and keeps punctuation with words
**WhitespaceSplit**: Only splits on whitespace

In [ ]:
text = "Hello, world! How are you?"

whitespace = pre_tokenizers.Whitespace()
whitespace_split = pre_tokenizers.WhitespaceSplit()

print(f"Text: {text}\n")

print("Whitespace (splits on spaces, keeps punctuation):")
for word, offset in whitespace.pre_tokenize_str(text):
    print(f"  '{word}'")

print("\nWhitespaceSplit (only splits on spaces):")
for word, offset in whitespace_split.pre_tokenize_str(text):
    print(f"  '{word}'")

## 2.3 ByteLevel Pre-tokenizer (GPT-2 style)

Converts all characters to bytes and splits on whitespace. Handles any Unicode character.

In [ ]:
byte_level = pre_tokenizers.ByteLevel(add_prefix_space=True)

examples = [
    "Hello world!",
    "café résumé",
    "日本語テキスト"  # Japanese text
]

print("ByteLevel Pre-tokenization:")
print("=" * 60)
for text in examples:
    pre_tokenized = byte_level.pre_tokenize_str(text)
    print(f"\nText: {text}")
    print("Tokens:")
    for word, offset in pre_tokenized:
        print(f"  '{word}'")

## 2.4 Punctuation Pre-tokenizer

Isolates punctuation from words.

In [ ]:
punctuation = pre_tokenizers.Punctuation(behavior="isolated")

examples = [
    "Hello, world!",
    "What's that?",
    "Cost: $99.99"
]

print("Punctuation Pre-tokenization (isolated):")
print("=" * 60)
for text in examples:
    pre_tokenized = punctuation.pre_tokenize_str(text)
    print(f"\nText: {text}")
    print("Tokens: ", end="")
    print([word for word, _ in pre_tokenized])

## 2.5 Digits Pre-tokenizer

Isolates digits from text.

In [ ]:
digits = pre_tokenizers.Digits(individual_digits=True)

examples = [
    "Room 101",
    "Price: $1234",
    "Year 2024"
]

print("Digits Pre-tokenization:")
print("=" * 60)
for text in examples:
    pre_tokenized = digits.pre_tokenize_str(text)
    print(f"\nText: {text}")
    print("Tokens: ", end="")
    print([word for word, _ in pre_tokenized])

## 2.6 Combining Pre-tokenizers (Sequence)

In [ ]:
# BERT-style: Whitespace + Punctuation
bert_pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation(behavior="isolated")
])

text = "Hello, world! How are you?"
pre_tokenized = bert_pre_tokenizer.pre_tokenize_str(text)

print(f"Text: {text}\n")
print("BERT-style Pre-tokenization (Whitespace + Punctuation):")
for word, offset in pre_tokenized:
    print(f"  '{word}' at {offset}")

## 2.7 CharDelimiterSplit (Split on Custom Character)

In [ ]:
# Split on pipes
pipe_splitter = pre_tokenizers.CharDelimiterSplit('|')

text = "apple|banana|cherry"
pre_tokenized = pipe_splitter.pre_tokenize_str(text)

print(f"Text: {text}\n")
print("Split on '|':")
for word, offset in pre_tokenized:
    print(f"  '{word}'")

---

# Part 3: Post-processing

Post-processing adds special tokens (like [CLS], [SEP]) after tokenization.

## 3.1 Template Processing (BERT-style)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.processors import TemplateProcessing

# Create simple tokenizer
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Add BERT-style post-processing
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", 1),
        ("[SEP]", 2),
    ],
)

# Train on simple data
from tokenizers.trainers import WordPieceTrainer
trainer = WordPieceTrainer(
    vocab_size=100,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)
tokenizer.train_from_iterator(["hello world", "how are you"] * 50, trainer)

# Test single sequence
output = tokenizer.encode("hello world")
print("Single sequence with [CLS] and [SEP]:")
print(f"Tokens: {output.tokens}")
print(f"IDs: {output.ids}")

# Test pair
output = tokenizer.encode("hello", "world")
print("\nSentence pair with [CLS] and [SEP]:")
print(f"Tokens: {output.tokens}")
print(f"Type IDs: {output.type_ids}")

---

# Part 4: Decoders

Decoders convert token IDs back to readable text.

## 4.1 ByteLevel Decoder (GPT-2 style)

In [ ]:
# Create tokenizer with ByteLevel
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
tokenizer.decoder = decoders.ByteLevel()

# Train
trainer = trainers.BpeTrainer(vocab_size=200, special_tokens=["<PAD>"])
tokenizer.train_from_iterator(["hello world", "machine learning"] * 50, trainer)

# Encode and decode
text = "Hello, world!"
output = tokenizer.encode(text)
decoded = tokenizer.decode(output.ids)

print(f"Original: {text}")
print(f"Tokens:   {output.tokens}")
print(f"Decoded:  {decoded}")

## 4.2 WordPiece Decoder (BERT style)

Removes `##` prefixes and joins subwords.

In [ ]:
# Simulate WordPiece tokens
tokens = ["running", "##ly", "fast", "##er"]

# Manual demonstration
print("WordPiece tokens: ", tokens)
print("\nDecoding process:")
decoded_words = []
current_word = ""

for token in tokens:
    if token.startswith("##"):
        current_word += token[2:]  # Remove ##
    else:
        if current_word:
            decoded_words.append(current_word)
        current_word = token

if current_word:
    decoded_words.append(current_word)

print(f"Decoded: {' '.join(decoded_words)}")

---

# Part 5: Complete Pipeline Examples

## 5.1 BERT-style Tokenizer (Complete Pipeline)

In [ ]:
# Create BERT-style tokenizer from scratch
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

# 1. Normalization
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFD(),
    normalizers.Lowercase(),
    normalizers.StripAccents()
])

# 2. Pre-tokenization
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation(behavior="isolated")
])

# 3. Post-processing
tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", 1), ("[SEP]", 2)]
)

# 4. Decoder
tokenizer.decoder = decoders.WordPiece(prefix="##")

# Train
trainer = trainers.WordPieceTrainer(
    vocab_size=500,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

training_data = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is awesome!",
    "Natural language processing is fascinating."
] * 100

tokenizer.train_from_iterator(training_data, trainer)

# Test
text = "HELLO, World! Machine Learning."
output = tokenizer.encode(text)

print("BERT-style Complete Pipeline:")
print("=" * 60)
print(f"Original:   {text}")
print(f"Normalized: {tokenizer.normalizer.normalize_str(text)}")
print(f"Tokens:     {output.tokens}")
print(f"Decoded:    {tokenizer.decode(output.ids)}")

## 5.2 GPT-2 style Tokenizer (Complete Pipeline)

In [ ]:
# Create GPT-2 style tokenizer
tokenizer = Tokenizer(BPE())

# 1. No normalization (GPT-2 is case-sensitive)
tokenizer.normalizer = None

# 2. ByteLevel pre-tokenization
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)

# 3. ByteLevel decoder
tokenizer.decoder = decoders.ByteLevel()

# Train
trainer = trainers.BpeTrainer(
    vocab_size=500,
    special_tokens=["<|endoftext|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

tokenizer.train_from_iterator(training_data, trainer)

# Test
text = "Hello, World! 🌍"
output = tokenizer.encode(text)

print("GPT-2 style Complete Pipeline:")
print("=" * 60)
print(f"Original: {text}")
print(f"Tokens:   {output.tokens}")
print(f"Decoded:  {tokenizer.decode(output.ids)}")

---

# Summary

## Pipeline Components

### 1. Normalization
- **NFD/NFKC**: Unicode normalization
- **Lowercase**: Convert to lowercase
- **StripAccents**: Remove accents
- **Replace**: Pattern replacement
- **Strip**: Remove whitespace
- **Sequence**: Chain normalizers

### 2. Pre-tokenization
- **Whitespace**: Split on spaces (keeps punctuation)
- **WhitespaceSplit**: Only split on spaces
- **ByteLevel**: Byte-level splitting (GPT-2)
- **Punctuation**: Isolate punctuation
- **Digits**: Isolate digits
- **CharDelimiterSplit**: Custom delimiter
- **Sequence**: Chain pre-tokenizers

### 3. Post-processing
- **TemplateProcessing**: Add special tokens ([CLS], [SEP])
- Configure for single or paired sequences
- Set type IDs for sentence pairs

### 4. Decoding
- **ByteLevel**: Decode byte-level tokens
- **WordPiece**: Join subwords (remove ##)
- **BPE**: Merge BPE tokens

## Common Patterns

**BERT**: NFD + Lowercase + StripAccents → WhitespaceSplit + Punctuation → WordPiece → [CLS]/[SEP]

**GPT-2**: No normalization → ByteLevel → BPE → ByteLevel decoder

**RoBERTa**: Similar to GPT-2 but with different special tokens

## Next Steps

- Experiment with different combinations
- Build custom pipelines for your domain
- Compare performance of different normalizers
- Understand trade-offs (speed vs accuracy)